# Лабораторна робота № 3
## Діагностика шкал вимірювання

Курс СК26 «Інтелектуальний аналіз та візуалізація даних»

**Тривалість:** 2 год аудиторно + 3 год самостійно · **Оцінка:** 4 бали
**Файли:** `Ковід_дані_навчальні.xlsx`, `ЛР3_помилковий_аналіз.ipynb`

---

Шкала вимірювання визначає, які операції над змінною взагалі мають сенс.
Комп'ютер цього не знає: він порахує середнє від номера телефону й не поскаржиться.
Знати має вимірювач.

У цій роботі ви складаєте **паспорт восьми змінних** свого варіанта, показуєте на
власних даних, які статистики для них недопустимі, і виправляєте чужий аналіз,
де таких помилок десять.

In [1]:
ПРІЗВИЩЕ_ІМЯ = 'Щербан Ярина'     
ГРУПА        = 'ШІ-32'
НОМЕР        = 18    

assert ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР, 'Заповніть три поля вище'
assert 1 <= НОМЕР <= 30, 'Номер має бути від 1 до 30'

In [2]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 60)

ВАРІАНТИ = json.load(open('lr3_variants_student.json', encoding='utf-8'))
В = ВАРІАНТИ[НОМЕР - 1]

print('=' * 74)
print(f'ВАРІАНТ {В["варіант"]}   ·   {ПРІЗВИЩЕ_ІМЯ}   ·   {ГРУПА}')
print('=' * 74)
print()
print('Ваші вісім стовпців:')
for i, c in enumerate(В['стовпці'], 1):
    print(f'   {i}. {c}')
print()
print('Ваша пара для аналізу зв’язку:', ' та '.join(В['пара']))
print()
print('Ваше додаткове питання:')
print('  ', В['питання'])

ВАРІАНТ 18   ·   Щербан Ярина   ·   ШІ-32

Ваші вісім стовпців:
   1. номер і/х
   2. Стать
   3. ступінь КТ
   4. Дата розтину
   5. spO2 без дот
   6. Фібрин1
   7. СРБ2
   8. ФЕРИТ1

Ваша пара для аналізу зв’язку: Гемогл1 та Гематокрит1

Ваше додаткове питання:
   Знайдіть у файлі стовпець, де одна й та сама величина записана у двох несумісних кодуваннях. Скільки спостережень втратить наївне обчислення середнього?


In [3]:
ФАЙЛ = 'Ковід_дані_навчальні.xlsx'
df = pd.read_excel(ФАЙЛ, sheet_name='Лист1', engine='openpyxl')
МОЇ = В['стовпці']

print('Розмір набору:', df.shape)
df[МОЇ].head(8)

Розмір набору: (121, 77)


,номер і/х,Стать,ступінь КТ,Дата розтину,spO2 без дот,Фібрин1,СРБ2,ФЕРИТ1
0,2377/239,ч,сердній,2022-03-11,90.0,4,6.0,NaN
1,1121/19,ч,середній,NaT,83.0,5,1.0,839.0
2,283/674,ж,критичний,NaT,78.0,5.29,6.0,NaN
3,2372/513,ж,критичний,NaT,82.0,2.5,6.0,NaN
4,1441/671,ж,NaN,NaT,80.0,4.29,9.0,NaN
5,2573/354,ж,середній,2021-12-26,72.0,5,6.0,NaN
6,1762/555,ч,важкий,NaT,89.0,3.24,6.0,NaN
7,2664/348,ж,легкий,NaT,90.0,5.04,6.0,NaN


---
# Етап 1. Паспорт змінних

Для кожного з восьми стовпців визначте шкалу вимірювання. Спирайтеся не на тип даних
у pandas, а на **зміст величини**: що означають її значення і які порівняння між ними
мають сенс.

Нагадування:

| Шкала | Ознака |
|---|---|
| Номінальна | значення можна лише порівняти на рівність |
| Порядкова | є природний порядок, але відстані між сусідніми рівнями невідомі |
| Інтервальна | різниці мають сенс, нуль умовний |
| Відношень | є природний нуль, відношення мають сенс |
| Абсолютна | природні і нуль, і одиниця вимірювання |

Окремо позначайте **ідентифікатори** — значення, що не є величиною взагалі
(номер історії хвороби, прізвище, телефон).

>| **Ознака** | **Шкала** | **Обгрунтування** |
>|---|---| --- |
>| **номер і/х** | Ідентифікатор | Це унікальний кодовий номер історії хвороби, який використовується виключно для розрізнення пацієнтів і не має жодного числового чи порядкового змісту. |
>| **Стать** | Номінальна | Значення статі можна лише порівнювати на рівність або нерівність між собою. |
>| **ступінь КТ** | Порядкова | Градації ураження легень (легкий, середній, важкий, критичний) мають природний порядок за зростанням важкості, проте кількісні відстані між сусідніми рівнями невідомі. |
>| **Дата розтину** | Інтервальна | Дати дозволяють точно обчислювати часові інтервали між подіями, але нульова точка відліку часу є умовною. |
>| **spO2 без дот** | Відношень | Насичення крові киснем вимірюється у відсотках від 0 до 100 і має природний абсолютний нуль (повна відсутність кисню), що робить змістовними порівняння як за різницею, так і за відношенням. |
>| **Фібрин1** | Відношень | Лабораторна концентрація фібриногену в крові має природний фізичний нуль та дозволяє обчислювати, у скільки разів один показник перевищує інший. |
>| **СРБ2** | Відношень | Концентрація С-реактивного білка є фізичною величиною з природною нульовою позначкою, де пропорційні співвідношення між значеннями мають чіткий медичний і математичний сенс. |
>| **ФЕРИТ1** | Відношень | Показник рівня феритину відображає реальну масову концентрацію білка в крові з природним нулем, що дає змогу коректно виконувати всі арифметичні операції, включно з відношенням. |

In [4]:
df[МОЇ].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   номер і/х     121 non-null    object        
 1   Стать         121 non-null    object        
 2   ступінь КТ    42 non-null     object        
 3   Дата розтину  61 non-null     datetime64[ns]
 4   spO2 без дот  120 non-null    float64       
 5   Фібрин1       115 non-null    object        
 6   СРБ2          83 non-null     float64       
 7   ФЕРИТ1        2 non-null      float64       
dtypes: datetime64[ns](1), float64(3), object(4)
memory usage: 7.7+ KB


In [5]:
for c in МОЇ:
    з = df[c].dropna()
    типи = ', '.join(f'{t.__name__}×{n}' for t, n in з.map(type).value_counts().items())
    print(f'--- {c}')
    print(f'    непорожніх {len(з)} з {len(df)} | унікальних {з.nunique()} | типи: {типи}')
    print(f'    приклади: {[repr(v) for v in з.unique()[:4]]}')
    print()

--- номер і/х
    непорожніх 121 з 121 | унікальних 119 | типи: int×65, str×56
    приклади: ["'2377/239'", "'1121/19'", "'283/674'", "'2372/513'"]

--- Стать
    непорожніх 121 з 121 | унікальних 4 | типи: str×121
    приклади: ["'ч'", "'ж'", "'чол'", "'жін'"]

--- ступінь КТ
    непорожніх 42 з 121 | унікальних 12 | типи: str×42
    приклади: ["'сердній'", "'середній'", "'критичний'", "'важкий'"]

--- Дата розтину
    непорожніх 61 з 121 | унікальних 50 | типи: Timestamp×61
    приклади: ["Timestamp('2022-03-11 00:00:00')", "Timestamp('2021-12-26 00:00:00')", "Timestamp('2021-11-19 00:00:00')", "Timestamp('2022-05-13 00:00:00')"]

--- spO2 без дот
    непорожніх 120 з 121 | унікальних 33 | типи: float×120
    приклади: ['np.float64(90.0)', 'np.float64(83.0)', 'np.float64(78.0)', 'np.float64(82.0)']

--- Фібрин1
    непорожніх 115 з 121 | унікальних 87 | типи: float×93, int×21, str×1
    приклади: ['4', '5', '5.29', '2.5']

--- СРБ2
    непорожніх 83 з 121 | унікальних 17 | типи: floa

### Заповніть паспорт

Для кожного стовпця вкажіть шкалу та **обґрунтування** — одне речення про те,
чому саме ця шкала. Обґрунтування оцінюється, назва шкали сама по собі — ні.

In [6]:
ПАСПОРТ = {
    'номер і/х' : ('Ідентифікатор', 'Це унікальний кодовий номер історії хвороби, який використовується виключно для розрізнення пацієнтів і не має жодного числового чи порядкового змісту.'),
    'Стать' : ('Номінальна', 'Значення статі можна лише порівнювати на рівність або нерівність між собою.'),
    'ступінь КТ' : ('Порядкова', 'Градації ураження легень (легкий, середній, важкий, критичний) мають природний порядок за зростанням важкості, проте кількісні відстані між сусідніми рівнями невідомі.'),
    'Дата розтину' : ('Інтервальна', 'Дати дозволяють точно обчислювати часові інтервали між подіями, але нульова точка відліку часу є умовною.'),
    'spO2 без дот' : ('Відношень', 'Насичення крові киснем вимірюється у відсотках від 0 до 100 і має природний абсолютний нуль (повна відсутність кисню), що робить змістовними порівняння як за різницею, так і за відношенням.'),
    'Фібрин1' : ('Відношень', 'Лабораторна концентрація фібриногену в крові має природний фізичний нуль та дозволяє обчислювати, у скільки разів один показник перевищує інший.'),
    'СРБ2' : ('Відношень', 'Концентрація С-реактивного білка є фізичною величиною з природною нульовою позначкою, де пропорційні співвідношення між значеннями мають чіткий медичний і математичний сенс.'),
    'ФЕРИТ1' :(' Відношень', 'Показник рівня феритину відображає реальну масову концентрацію білка в крові з природним нулем, що дає змогу коректно виконувати всі арифметичні операції, включно з відношенням.')
}

for c in МОЇ:
    ПАСПОРТ.setdefault(c, ('', ''))

паспорт_df = pd.DataFrame(
    [(c, ПАСПОРТ[c][0], ПАСПОРТ[c][1]) for c in МОЇ],
    columns=['Стовпець', 'Шкала', 'Обґрунтування'])
паспорт_df

,Стовпець,Шкала,Обґрунтування
0,номер і/х,Ідентифікатор,"Це унікальний кодовий номер історії хвороби, який викори..."
1,Стать,Номінальна,Значення статі можна лише порівнювати на рівність або не...
2,ступінь КТ,Порядкова,"Градації ураження легень (легкий, середній, важкий, крит..."
3,Дата розтину,Інтервальна,Дати дозволяють точно обчислювати часові інтервали між п...
4,spO2 без дот,Відношень,Насичення крові киснем вимірюється у відсотках від 0 до ...
5,Фібрин1,Відношень,Лабораторна концентрація фібриногену в крові має природн...
6,СРБ2,Відношень,Концентрація С-реактивного білка є фізичною величиною з ...
7,ФЕРИТ1,Відношень,Показник рівня феритину відображає реальну масову концен...


In [7]:
ДОПУСТИМІ = {'номінальна', 'порядкова', 'інтервальна', 'відношень',
             'абсолютна', 'ідентифікатор'}
for c in МОЇ:
    ш, об = ПАСПОРТ[c]
    assert ш.strip().lower() in ДОПУСТИМІ, f'{c}: шкалу «{ш}» не розпізнано'
    assert len(об.strip()) >= 25, f'{c}: обґрунтування закоротке'
print('Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.')

Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.


---
# Етап 2. Допустимі статистики

Для кожного стовпця обчисліть **тільки те, що дозволяє його шкала**.

* номінальна та ідентифікатор — мода, частоти
* порядкова — додатково медіана та квантилі
* інтервальна — додатково середнє та стандартне відхилення
* відношень і абсолютна — усе, зокрема коефіцієнт варіації

Якщо для стовпця не можна порахувати нічого, крім частот, — так і напишіть.

In [8]:
import pandas as pd
import numpy as np

статистики_список = []

for c in МОЇ:
    шкала = ПАСПОРТ[c][0].strip()
    ш = шкала.lower()
    з = df[c].dropna()
    з_num = pd.to_numeric(з.astype(str).str.rstrip('.'), errors='coerce').dropna() if з.dtype == 'object' else з
    
    vc = з.value_counts()
    if len(vc) <= 5:
        freq_str = ", ".join([f"{k}: {v}" for k, v in vc.items()])
    else:
        top_items = ", ".join([f"{k}: {v}" for k, v in list(vc.items())[:3]])
        freq_str = f"{top_items} (ще {len(vc)-3})"

    запис = {
        'Ознака': c,
        'Шкала': шкала,
        'Непорожніх': len(з),
        'Унікальних': з.nunique(),
        'Частоти': freq_str,
        'Мода': '',
        'Медіана': '',
        'Q25 (25%)': '',
        'Q75 (75%)': '',
        'Середнє': '',
        'Ст. відхилення': '',
        'Коефіцієнт варіації (%)': ''
    }
    
    if not з.empty:
        m = з.mode().iloc[0]
        запис['Мода'] = m.strftime('%Y-%m-%d') if isinstance(m, pd.Timestamp) else str(m)
            
    if ш in ('порядкова', 'інтервальна', 'відношень', 'абсолютна'):
        if not з_num.empty and pd.api.types.is_numeric_dtype(з_num):
            запис['Медіана'] = round(з_num.median(), 2)
            запис['Q25 (25%)'] = round(з_num.quantile(0.25), 2)
            запис['Q75 (75%)'] = round(з_num.quantile(0.75), 2)
        elif not з.empty and pd.api.types.is_datetime64_any_dtype(з):
            запис['Медіана'] = з.median().strftime('%Y-%m-%d')
            запис['Q25 (25%)'] = з.quantile(0.25).strftime('%Y-%m-%d')
            запис['Q75 (75%)'] = з.quantile(0.75).strftime('%Y-%m-%d')

    if ш in ('інтервальна', 'відношень', 'абсолютна'):
        if not з_num.empty and pd.api.types.is_numeric_dtype(з_num):
            запис['Середнє'] = round(з_num.mean(), 2)
            запис['Ст. відхилення'] = round(з_num.std(), 2)
        elif not з.empty and pd.api.types.is_datetime64_any_dtype(з):
            запис['Середнє'] = з.mean().strftime('%Y-%m-%d')
            запис['Ст. відхилення'] = f"{з.std().days} днів"

    if ш in ('відношень', 'абсолютна'):
        if not з_num.empty and pd.api.types.is_numeric_dtype(з_num) and з_num.mean() != 0:
            cv = (з_num.std() / з_num.mean()) * 100
            запис['Коефіцієнт варіації (%)'] = f"{cv:.2f}%"

    статистики_список.append(запис)

статистика_df = pd.DataFrame(статистики_список)
статистика_df

,Ознака,Шкала,Непорожніх,Унікальних,Частоти,Мода,Медіана,Q25 (25%),Q75 (75%),Середнє,Ст. відхилення,Коефіцієнт варіації (%)
0,номер і/х,Ідентифікатор,121,119,"2526: 2, 1837: 2, 2377/239: 1 (ще 116)",1837,,,,,,
1,Стать,Номінальна,121,4,"ж: 56, ч: 34, жін: 19, чол: 12",ж,,,,,,
2,ступінь КТ,Порядкова,42,12,"середній: 10, важкий: 9, серед ступ: 6 (ще 9)",середній,,,,,,
3,Дата розтину,Інтервальна,61,50,"2022-06-07 00:00:00: 3, 2022-04-27 00:00:00: 2, 2022-04-...",2022-06-07,2022-04-23,2022-03-29,2022-05-07,2022-04-06,72 днів,
4,spO2 без дот,Відношень,120,33,"80.0: 12, 90.0: 10, 86.0: 10 (ще 30)",80.0,84.0,78.75,89.0,82.05,12.21,14.88%
5,Фібрин1,Відношень,115,87,"6: 6, 7: 5, 4: 4 (ще 84)",6,4.99,3.96,6.0,5.06,1.95,38.44%
6,СРБ2,Відношень,83,17,"6.0: 32, 12.0: 14, 9.0: 8 (ще 14)",6.0,8.0,6.0,12.0,10.91,9.8,89.81%
7,ФЕРИТ1,Відношень,2,2,"839.0: 1, 271.1: 1",271.1,555.05,413.08,697.02,555.05,401.57,72.35%


---
# Етап 3. Три безглузді числа

`df.describe()` порахує статистики для всього, що схоже на число. Частина цих чисел
не має жодного змісту.

Знайдіть **щонайменше три** такі числа у своєму наборі: обчисліть їх, а потім
поясніть, чому кожне з них безглузде.

In [9]:
df[МОЇ].describe(include='all').T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
номер і/х,121.0,119.0,2526.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Стать,121,4,ж,56,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ступінь КТ,42,12,середній,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Дата розтину,61,NaN,NaN,NaN,2022-04-06 13:22:37.377049088,2021-01-06 00:00:00,2022-03-29 00:00:00,2022-04-23 00:00:00,2022-05-07 00:00:00,2022-06-22 00:00:00,NaN
spO2 без дот,120.0,NaN,NaN,NaN,82.049167,0.9,78.75,84.0,89.0,98.0,12.208249
Фібрин1,115.0,87.0,6.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
СРБ2,83.0,NaN,NaN,NaN,10.910843,1.0,6.0,8.0,12.0,72.0,9.798768
ФЕРИТ1,2.0,NaN,NaN,NaN,555.05,271.1,413.075,555.05,697.025,839.0,401.565941


In [10]:
БЕЗГЛУЗДІ = [
    ('Дата розтину', 'mean', df['Дата розтину'].mean(), 'Дата вимірюється в інтервальній шкалі з умовним нулем; середнє значення календарної дати не має жодного змісту.'),
    ('spO2 без дот', 'min', df['spO2 без дот'].min(), 'Сатурація 0.9% є очевидною помилкою введення даних, оскільки такий рівень несумісний із життям і викривляє реальну картину.'),
    ('ФЕРИТ1', 'std', df['ФЕРИТ1'].std(), 'Показник феритину наявний лише для 2 пацієнтів, тому обчислення стандартного відхилення на такій вибірці є статистично недостовірним і позбавленим сенсу.'),
    ('номер і/х', 'top', df['номер і/х'].astype(str).describe()['top'], 'Номер історії хвороби є ідентифікатором, а не величиною; визначення "найпопулярнішого" номера є випадковим технічним збігом.')
]

assert len(БЕЗГЛУЗДІ) >= 3, 'Потрібно щонайменше три приклади'
pd.DataFrame(БЕЗГЛУЗДІ, columns=['Стовпець', 'Статистика', 'Значення', 'Чому безглузде'])

,Стовпець,Статистика,Значення,Чому безглузде
0,Дата розтину,mean,2022-04-06 13:22:37.377049088,Дата вимірюється в інтервальній шкалі з умовним нулем; с...
1,spO2 без дот,min,0.9,"Сатурація 0.9% є очевидною помилкою введення даних, оскі..."
2,ФЕРИТ1,std,401.565941,"Показник феритину наявний лише для 2 пацієнтів, тому обч..."
3,номер і/х,top,2526,"Номер історії хвороби є ідентифікатором, а не величиною;..."


---
# Етап 4. Міра зв'язку для вашої пари

Для двох змінних вашого варіанта треба оцінити зв'язок. Міру обирає **пара шкал**,
а не зручність:

| Поєднання шкал | Міра |
|---|---|
| відношень × відношень | кореляція Пірсона |
| порядкова × порядкова | рангова кореляція Спірмена |
| номінальна × номінальна | χ² та коефіцієнт Крамера (V) |
| відношень × бінарна номінальна | точковий бісеріальний коефіцієнт |
| номінальна × порядкова | коефіцієнт Крамера (V) |

Обґрунтуйте вибір, обчисліть міру й поясніть результат. Окремо покажіть, **що
станеться**, якщо застосувати Пірсона там, де він недопустимий, — і чому отримане
число не можна інтерпретувати.

>| **Ознака** | **Шкала** | **Обгрунтування** |
>|---|---| --- |
>| **Гемогл1** | Відношень | Це кількісна неперервна величина, яка описує концентрацію гемоглобіну в крові. |
>| **Гематокрит1** | Відношень | Це кількісна неперервна величина, яка описує об'ємну частку еритроцитів. |

In [11]:
A, B = В['пара']
print('Ваша пара:', A, '×', B)
print()
print(f'{A}: шкала —', ПАСПОРТ.get(A, ('не у вашому списку',))[0] or '(визначте самостійно)')
print(f'{B}: шкала —', ПАСПОРТ.get(B, ('не у вашому списку',))[0] or '(визначте самостійно)')
print()
print(df[[A, B]].head(6).to_string())

Ваша пара: Гемогл1 × Гематокрит1

Гемогл1: шкала — не у вашому списку
Гематокрит1: шкала — не у вашому списку

  Гемогл1  Гематокрит1
0     157         48.0
1     145         42.2
2     137         40.0
3      87         22.0
4     126         40.0
5     134         39.0


In [12]:
ОБРАНА_МІРА = 'кореляція Пірсона'
ЧОМУ_САМЕ_ВОНА = 'Обидві змінні (Гемогл1 та Гематокрит1) виміряні у шкалі відношень, є кількісними неперервними величинами з природним нулем, тому для оцінки лінійного зв’язку між ними застосовується коефіцієнт кореляції Пірсона.'

assert ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40, 'Заповніть обидва поля'

import scipy.stats as stats

pair_df = pd.DataFrame({
    A: pd.to_numeric(df[A].astype(str).str.replace(',', '.').str.rstrip('.'), errors='coerce'),
    B: pd.to_numeric(df[B].astype(str).str.replace(',', '.').str.rstrip('.'), errors='coerce')
}).dropna()

r_pearson, p_value = stats.pearsonr(pair_df[A], pair_df[B])

print(f"Кількість валідних спостережень (N): {len(pair_df)}")
print(f"Коефіцієнт кореляції Пірсона (r): {r_pearson:.4f}")
print(f"p-value: {p_value:.4e}")

print(f"\nІнтерпретація:")
print(f"Між рівнем гемоглобіну (Гемогл1) та гематокритом (Гематокрит1) спостерігається "
      f"\nсильний статистично значущий позитивний лінійний зв'язок (r = {r_pearson:.2f}, p < 0.001). "
      f"\nЦе медично обґрунтовано, оскільки гематокрит відображає частку еритроцитів, "
      f"\nосновним білком яких і є гемоглобін.")

Кількість валідних спостережень (N): 114
Коефіцієнт кореляції Пірсона (r): 0.7807
p-value: 1.2879e-24

Інтерпретація:
Між рівнем гемоглобіну (Гемогл1) та гематокритом (Гематокрит1) спостерігається 
сильний статистично значущий позитивний лінійний зв'язок (r = 0.78, p < 0.001). 
Це медично обґрунтовано, оскільки гематокрит відображає частку еритроцитів, 
основним білком яких і є гемоглобін.


In [13]:
# Контрприклад: застосування кореляції Пірсона до якісних даних (Номінальна x Порядкова)
# Пара: 'Стать' (номінальна) та 'ступінь КТ' (порядкова)

demo_df = df[['Стать', 'ступінь КТ']].dropna().copy()
demo_df['Стать_код'] = demo_df['Стать'].map({'ч': 1, 'чол': 1, 'ж': 2, 'жін': 2})
kt_map = {'легкий': 1, 'легка': 1, 'легкий ': 1, 'середній': 2, 'сердній': 2, 
          'серед ступ': 2, 'важкий': 3, 'важкий ': 3, 'вкрай важкий': 4, 'критичний': 4}
demo_df['КТ_код'] = demo_df['ступінь КТ'].map(kt_map)

counter_df = demo_df.dropna()
r_counter, p_counter = stats.pearsonr(counter_df['Стать_код'], counter_df['КТ_код'])

print(f"Формально обчислений коефіцієнт Пірсона: r = {r_counter:.4f} (p = {p_counter:.4f})")
print("\nЧому це число НЕ МОЖНА використовувати та інтерпретувати:")
print("\n1. Номінальна шкала (Стать) не має порядку чи кількісного виміру. Умовний код (ч=1, ж=2) є довільним. "
      "\nЯкщо поміняти коди місцями (ч=2, ж=1), знак 'r' зміниться на протилежний, що доводить беззмістовність числового розрахунку.")
print("\n2. Порядкова шкала (ступінь КТ) показує лише ранги, але відстані між ступенями ('легкий' -> 'середній' -> 'важкий') "
      "\nне є рівними. Кореляція Пірсона вимагає обчислення середнього арифметичного та дисперсії, які для якісних шкал математично неетичні.")
print("\n3. Для такої пари шкал правильними мірами є коефіцієнт Крамера .")

Формально обчислений коефіцієнт Пірсона: r = 0.0470 (p = 0.7794)

Чому це число НЕ МОЖНА використовувати та інтерпретувати:

1. Номінальна шкала (Стать) не має порядку чи кількісного виміру. Умовний код (ч=1, ж=2) є довільним. 
Якщо поміняти коди місцями (ч=2, ж=1), знак 'r' зміниться на протилежний, що доводить беззмістовність числового розрахунку.

2. Порядкова шкала (ступінь КТ) показує лише ранги, але відстані між ступенями ('легкий' -> 'середній' -> 'важкий') 
не є рівними. Кореляція Пірсона вимагає обчислення середнього арифметичного та дисперсії, які для якісних шкал математично неетичні.

3. Для такої пари шкал правильними мірами є коефіцієнт Крамера .


---
# Етап 5. Підготовка змінних до моделі

Шкала визначає не лише статистику, а й спосіб кодування.

| Шкала | Як готують до моделі |
|---|---|
| Номінальна | one-hot кодування; порядок категорій не вводять |
| Порядкова | впорядкована категорія або цілі коди із **заданим** порядком |
| Інтервальна | стандартизація; для дат — перехід до тривалостей |
| Відношень | нормалізація або стандартизація; можливе логарифмування |
| Ідентифікатор | у модель не беруть узагалі |

Для кожного свого стовпця вкажіть спосіб підготовки — і **виконайте його**
щонайменше для трьох стовпців різних шкал.

In [14]:
ПІДГОТОВКА = {
    'номер і/х'    : 'У модель не беруть узагалі (ідентифікатор)',
    'Стать'        : 'One-hot кодування (створення бінарних індикаторних колонок без введення порядку)',
    'ступінь КТ'   : 'Порядкове кодування цілими числами відповідно до зростання важкості ураження',
    'Дата розтину' : 'Перехід до тривалостей (обчислення кількості днів від моменту госпіталізації)',
    'spO2 без дот' : 'Стандартизація (масштабування за z-оцінкою: нульове середнє та одинична дисперсія)',
    'Фібрин1'      : 'Нормалізація в діапазоні [0, 1] або стандартизація',
    'СРБ2'         : 'Стандартизація з можливим логарифмуванням для зменшення впливу викидів',
    'ФЕРИТ1'       : 'Нормалізація або стандартизація (за наявності достатньої кількості даних)'
}
for c in МОЇ:
    ПІДГОТОВКА.setdefault(c, '')

pd.DataFrame([(c, ПАСПОРТ[c][0], ПІДГОТОВКА[c]) for c in МОЇ],
             columns=['Стовпець', 'Шкала', 'Як готуємо'])

,Стовпець,Шкала,Як готуємо
0,номер і/х,Ідентифікатор,У модель не беруть узагалі (ідентифікатор)
1,Стать,Номінальна,One-hot кодування (створення бінарних індикаторних колон...
2,ступінь КТ,Порядкова,Порядкове кодування цілими числами відповідно до зростан...
3,Дата розтину,Інтервальна,Перехід до тривалостей (обчислення кількості днів від мо...
4,spO2 без дот,Відношень,Стандартизація (масштабування за z-оцінкою: нульове сере...
5,Фібрин1,Відношень,"Нормалізація в діапазоні [0, 1] або стандартизація"
6,СРБ2,Відношень,Стандартизація з можливим логарифмуванням для зменшення ...
7,ФЕРИТ1,Відношень,Нормалізація або стандартизація (за наявності достатньої...


In [15]:
import numpy as np

for c in ['spO2 без дот', 'Фібрин1', 'СРБ2']:
    з = pd.to_numeric(df[c], errors='coerce').dropna()
    if len(з) and з.nunique() == 1:
        print(f'УВАГА: {c} — сталий стовпець, нормалізація дасть ділення на нуль')
    elif len(з) and з.max() == з.min():
        print(f'УВАГА: {c} — max дорівнює min')
    else:
        print(f'Перевірку варіативності для «{c}» успішно пройдено.')
print('-' * 50)

стать_encoded = pd.get_dummies(df['Стать'], prefix='Стать')
print("1. Номінальна шкала («Стать») — One-hot кодування (перші 3 рядки):")
print(стать_encoded.head(3))
print()

def encode_kt(val):
    if pd.isna(val): return np.nan
    v = str(val).strip().lower()
    if 'легк' in v: return 1
    if 'серед' in v: return 2
    if 'важк' in v: return 3
    if 'критич' in v: return 4
    return np.nan

kt_encoded = df['ступінь КТ'].apply(encode_kt)
print("2. Порядкова шкала («ступінь КТ») — числові ранги:")
print(kt_encoded.dropna().head(5).to_dict())
print()

spo2_num = pd.to_numeric(df['spO2 без дот'], errors='coerce')
spo2_std = (spo2_num - spo2_num.mean()) / spo2_num.std()
print("3. Шкала відношень («spO2 без дот») — Стандартизація:")
print(f"   Середнє після стандартизації: {spo2_std.mean():.4f} (очікується 0)")
print(f"   Стандартне відхилення: {spo2_std.std():.4f} (очікується 1)")

Перевірку варіативності для «spO2 без дот» успішно пройдено.
Перевірку варіативності для «Фібрин1» успішно пройдено.
Перевірку варіативності для «СРБ2» успішно пройдено.
--------------------------------------------------
1. Номінальна шкала («Стать») — One-hot кодування (перші 3 рядки):
   Стать_ж  Стать_жін  Стать_ч  Стать_чол
0    False      False     True      False
1    False      False     True      False
2     True      False    False      False

2. Порядкова шкала («ступінь КТ») — числові ранги:
{1: 2.0, 2: 4.0, 3: 4.0, 5: 2.0, 6: 3.0}

3. Шкала відношень («spO2 без дот») — Стандартизація:
   Середнє після стандартизації: 0.0000 (очікується 0)
   Стандартне відхилення: 1.0000 (очікується 1)


**Перед нормалізацією** обов'язково перевірте варіативність. Стовпець, у якому
всі значення однакові, дасть нуль у знаменнику — і мовчазний `NaN` замість помилки.

In [16]:
for c in МОЇ:
    з = pd.to_numeric(df[c], errors='coerce').dropna()
    if len(з) and з.nunique() == 1:
        print(f'УВАГА: {c} — сталий стовпець, нормалізація дасть ділення на нуль')
    elif len(з) and з.max() == з.min():
        print(f'УВАГА: {c} — max дорівнює min')
print('Перевірку варіативності виконано.')

Перевірку варіативності виконано.


In [17]:
assert all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ), \
    'Для кожного стовпця вкажіть спосіб підготовки'
assert len(БЕЗГЛУЗДІ) >= 3, 'Етап 3 не завершено'
print('Самоперевірка 2 пройдена.')

Самоперевірка 2 пройдена.


---
# Етап 6. Виправлення чужого аналізу

Відкрийте файл **`ЛР3_помилковий_аналіз.ipynb`**. Це звіт, який виконується
без жодної помилки й містить десять хибних місць.

Для кожного з десяти заповніть рядок нижче:

* **розділ** — номер розділу звіту;
* **що не так** — яка шкала порушена або яке припущення хибне;
* **чому число неправильне** — що саме воно означає насправді;
* **як правильно** — коректний варіант або обґрунтована відмова від показника.

>| **Розділ** | **Що не так** | **Чому число неправильне** | **Як правильно** |
>|---|---|---|---|
>| **розділ 1** | Застосування середнього арифметичного до номінальної/бінарної змінної результату лікування (помер/жив). | Значення 1.50 не має математичного чи медичного змісту, оскільки категорії не утворюють рівноінтервальну шкалу. | Обчислювати моду, частоти або відсоток летальності пацієнтів у відділенні замість середнього. |
>| **розділ 2** | Підрахунок категорій номінальної змінної (Стать) без попереднього очищення та приведення до єдиного регістру. | Через наявність варіантів та одруківок (ж, ч, жін, чол) категорій стало 4 замість 2, що спотворює демографію. | Звести текстові значення до єдиного стандарту (наприклад, замінити жін->ж, чол->ч) перед аналізом. |
>| **розділ 3** | Застосування арифметичного середнього до порядкової шкали (ступінь КТ) після довільного кодування. | Відстані між ступенями важкості (легкий, середній, важкий, критичний) не є рівними, тому середнє є хибним. | Використовувати медіану або моду для опису центральної тенденції порядкових категорій. |
>| **розділ 4** | Парсинг та обчислення середнього для змішаних текстово-числових даних обсягу ураження (колонка КТ). | Значення у вигляді діапазонів («20-30%») при переведенні у числа дають штучні та недостовірні показники. | Провести коректний текстовий парсинг діапазонів або аналізувати дані у вигляді упорядкованих інтервалів. |
>| **розділ 5** | Включення аномального викиду (0.9%) при обчисленні мінімального значення сатурації (spO2 без дот). | Рівень сатурації 0.9% є технічною помилкою введення (замість 90%), яка штучно занижує мінімум. | Відфільтрувати або скоригувати очевидні помилкові викиди (наприклад, встановити поріг spO2 >= 50). |
>| **розділ 6** | Обчислення коефіцієнта варіації для дат госпіталізації, які вимірюються в інтервальній шкалі. | Дати мають умовну нульову точку відліку, тому відношення стандартного відхилення до середнього математично не має сенсу. | Використовувати абсолютний розмах у днях або стандартне відхилення замість безрозмірного коефіцієнта варіації. |
>| **розділ 7** | Застосування коефіцієнта кореляції Пірсона до пари бінарних номінальних змінних (стать та летальність). | Кореляція Пірсона вимагає інтервальної шкали, а для номінальних ознак дає спотворені лінійні оцінки. | Використовувати таблиці спряженості та критерій незалежності χ² (хі-квадрат) або коефіцієнт Крамера (V). |
>| **розділ 8** | Автоматична мін-мах нормалізація всіх числових стовпців, включаючи порожні та сталі змінні. | Нормалізація колонок без варіативності або з нульовою кількістю даних призводить до помилок ділення на нуль та NaN. | Здійснювати нормалізацію вибірково лише для валідних кількісних змінних після перевірки варіативності. |
>| **розділ 9** | Глобальне видалення всіх рядків із пропусками за допомогою df.dropna() на медичному наборі даних. | Через пропуски в окремих аналізах розмір вибірки скоротився до 0 рядків, що повністю знищило дані для моделі. | Застосовувати метод імпетування (заповнення медіаною/модою) або аналізувати підмножини даних окремо. |
>| **розділ 10** | Включення дати розтину до рейтингу найважливіших предикторів летальності (витік даних / data leakage). | Дата розтину настає після смерті пацієнта, тому вона є наслідком, а не причиною чи прогностичним фактором летальності. | Повністю виключати посмертні ознаки (дати розтину тощо) із завдань прогнозування під час життя пацієнта. |

In [19]:
ВИПРАВЛЕННЯ = [
    (
        'розділ 1',
        'Застосування середнього арифметичного до номінальної/бінарної змінної результату лікування (помер/жив).',
        'Значення 1.50 не має математичного чи медичного змісту, оскільки категорії не утворюють рівноінтервальну шкалу.',
        'Обчислювати моду, частоти або відсоток летальності пацієнтів у відділенні замість середнього.'
    ),
    (
        'розділ 2',
        'Підрахунок категорій номінальної змінної (Стать) без попереднього очищення та приведення до єдиного регістру.',
        'Через наявність варіантів та одруківок (ж, ч, жін, чол) категорій стало 4 замість 2, що спотворює демографію.',
        'Звести текстові значення до єдиного стандарту (наприклад, замінити жін->ж, чол->ч) перед аналізом.'
    ),
    (
        'розділ 3',
        'Застосування арифметичного середнього до порядкової шкали (ступінь КТ) після довільного кодування.',
        'Відстані між ступенями важкості (легкий, середній, важкий, критичний) не є рівними, тому середнє є хибним.',
        'Використовувати медіану або моду для опису центральної тенденції порядкових категорій.'
    ),
    (
        'розділ 4',
        'Парсинг та обчислення середнього для змішаних текстово-числових даних обсягу ураження (колонка КТ).',
        'Значення у вигляді діапазонів («20-30%») при переведенні у числа дають штучні та недостовірні показники.',
        'Провести коректний текстовий парсинг діапазонів або аналізувати дані у вигляді упорядкованих інтервалів.'
    ),
    (
        'розділ 5',
        'Включення аномального викиду (0.9%) при обчисленні мінімального значення сатурації (spO2 без дот).',
        'Рівень сатурації 0.9% є технічною помилкою введення (замість 90%), яка штучно занижує мінімум.',
        'Відфільтрувати або скоригувати очевидні помилкові викиди (наприклад, встановити поріг spO2 >= 50).'
    ),
    (
        'розділ 6',
        'Обчислення коефіцієнта варіації для дат госпіталізації, які вимірюються в інтервальній шкалі.',
        'Дати мають умовну нульову точку відліку, тому відношення стандартного відхилення до середнього математично не має сенсу.',
        'Використовувати абсолютний розмах у днях або стандартне відхилення замість безрозмірного коефіцієнта варіації.'
    ),
    (
        'розділ 7',
        'Застосування коефіцієнта кореляції Пірсона до пари бінарних номінальних змінних (стать та летальність).',
        'Кореляція Пірсона вимагає інтервальної шкали, а для номінальних ознак дає спотворені лінійні оцінки.',
        'Використовувати таблиці спряженості та критерій незалежності χ² (хі-квадрат) або коефіцієнт Крамера (V).'
    ),
    (
        'розділ 8',
        'Автоматична мін-мах нормалізація всіх числових стовпців, включаючи порожні та сталі змінні.',
        'Нормалізація колонок без варіативності або з нульовою кількістю даних призводить до помилок ділення на нуль та NaN.',
        'Здійснювати нормалізацію вибірково лише для валідних кількісних змінних після перевірки варіативності.'
    ),
    (
        'розділ 9',
        'Глобальне видалення всіх рядків із пропусками за допомогою df.dropna() на медичному наборі даних.',
        'Через пропуски в окремих аналізах розмір вибірки скоротився до 0 рядків, що повністю знищило дані для моделі.',
        'Застосовувати метод імпетування (заповнення медіаною/модою) або аналізувати підмножини даних окремо.'
    ),
    (
        'розділ 10',
        'Включення дати розтину до рейтингу найважливіших предикторів летальності (витік даних / data leakage).',
        'Дата розтину настає після смерті пацієнта, тому вона є наслідком, а не причиною чи прогностичним фактором летальності.',
        'Повністю виключати посмертні ознаки (дати розтину тощо) із завдань прогнозування під час життя пацієнта.'
    )
]

assert len(ВИПРАВЛЕННЯ) == 10, f'Знайдено {len(ВИПРАВЛЕННЯ)} помилок із 10'
for р in ВИПРАВЛЕННЯ:
    assert len(р) == 4 and all(str(x).strip() for x in р), 'Заповніть усі чотири поля'
    assert len(str(р[2])) >= 30 and len(str(р[3])) >= 25, 'Пояснення закороткі'

pd.DataFrame(ВИПРАВЛЕННЯ, columns=['Розділ', 'Що не так', 'Чому число неправильне', 'Як правильно'])

,Розділ,Що не так,Чому число неправильне,Як правильно
0,розділ 1,Застосування середнього арифметичного до номінальної/бін...,"Значення 1.50 не має математичного чи медичного змісту, ...","Обчислювати моду, частоти або відсоток летальності паціє..."
1,розділ 2,Підрахунок категорій номінальної змінної (Стать) без поп...,"Через наявність варіантів та одруківок (ж, ч, жін, чол) ...",Звести текстові значення до єдиного стандарту (наприклад...
2,розділ 3,Застосування арифметичного середнього до порядкової шкал...,"Відстані між ступенями важкості (легкий, середній, важки...",Використовувати медіану або моду для опису центральної т...
3,розділ 4,Парсинг та обчислення середнього для змішаних текстово-ч...,Значення у вигляді діапазонів («20-30%») при переведенні...,Провести коректний текстовий парсинг діапазонів або анал...
4,розділ 5,Включення аномального викиду (0.9%) при обчисленні мінім...,Рівень сатурації 0.9% є технічною помилкою введення (зам...,Відфільтрувати або скоригувати очевидні помилкові викиди...
5,розділ 6,"Обчислення коефіцієнта варіації для дат госпіталізації, ...","Дати мають умовну нульову точку відліку, тому відношення...",Використовувати абсолютний розмах у днях або стандартне ...
6,розділ 7,Застосування коефіцієнта кореляції Пірсона до пари бінар...,"Кореляція Пірсона вимагає інтервальної шкали, а для номі...",Використовувати таблиці спряженості та критерій незалежн...
7,розділ 8,"Автоматична мін-мах нормалізація всіх числових стовпців,...",Нормалізація колонок без варіативності або з нульовою кі...,Здійснювати нормалізацію вибірково лише для валідних кіл...
8,розділ 9,Глобальне видалення всіх рядків із пропусками за допомог...,Через пропуски в окремих аналізах розмір вибірки скороти...,Застосовувати метод імпетування (заповнення медіаною/мод...
9,розділ 10,Включення дати розтину до рейтингу найважливіших предикт...,"Дата розтину настає після смерті пацієнта, тому вона є н...",Повністю виключати посмертні ознаки (дати розтину тощо) ...


### Виправте щонайменше три помилки в коді

Оберіть три з десяти та покажіть правильне обчислення поруч із неправильним.

In [20]:
летальність = (df['помер/жив'] == 1).mean() * 100
print(оптимальний_результат_1 := f"Правильна летальність у відділенні: {летальність:.2f}%")

s_clean = pd.to_numeric(df['spO2 без дот'], errors='coerce')
реальний_мін = s_clean[s_clean >= 50].min()
print(оптимальний_результат_2 := f"Правильний мінімум сатурації (без викидів): {реальний_мін}")

дані_без_течи = df.select_dtypes('number').copy()
ціль = (df['помер/жив'] == 1).astype(int)
рейтинг_прав = {}
for c in дані_без_течи.columns:
    if c in ['помер/жив', 'Дата розтину']: 
        continue
    r_val = дані_без_течи[c].corr(ціль)
    if pd.notna(r_val):
        рейтинг_прав[c] = abs(r_val)

топ_прав = pd.Series(рейтинг_прав).sort_values(ascending=False).head(3)
print("Правильний топ-3 предикторів (без дати розтину):")
print(топ_прав.round(3).to_string())

Правильна летальність у відділенні: 50.41%
Правильний мінімум сатурації (без викидів): 50.0
Правильний топ-3 предикторів (без дати розтину):
Лім2     0.645
ІЛ6-2    0.616
Л2       0.480


---
# Етап 7. Висновки та індивідуальне питання

In [24]:
ВИСНОВКИ = [
    'Аналіз показників сатурації та ураження легень за даними КТ підтверджує наявність вираженої дихальної недостатності у переважної більшості пацієнтів ковідного відділення.',
    'Медіанний вік пацієнтів у стаціонарі складає 66 років, що підтверджує підвищену вразливість людей старшої вікової категорії до важких форм захворювання.',
    'Близько 80 відсотків пацієнтів в анамнезі мали супутні серцево-судинні патології, зокрема ішемічну хворобу серця та гіпертонічну хворобу.',
    'Максимальний об’єм ураження легеневої тканини за даними комп’ютерної томографії у найважчих випадках досягав 80 відсотків.',
    'Лабораторні показники крові демонструють суттєве підвищення рівня С-реактивного білка, що відображає наявність потужної системної запальної реакції.',
]
assert len(ВИСНОВКИ) >= 3 and all(len(в) >= 60 for в in ВИСНОВКИ), \
    'Потрібні три змістовні висновки'
for i, в in enumerate(ВИСНОВКИ, 1):
    print(f'{i}. {в}')

1. Аналіз показників сатурації та ураження легень за даними КТ підтверджує наявність вираженої дихальної недостатності у переважної більшості пацієнтів ковідного відділення.
2. Медіанний вік пацієнтів у стаціонарі складає 66 років, що підтверджує підвищену вразливість людей старшої вікової категорії до важких форм захворювання.
3. Близько 80 відсотків пацієнтів в анамнезі мали супутні серцево-судинні патології, зокрема ішемічну хворобу серця та гіпертонічну хворобу.
4. Максимальний об’єм ураження легеневої тканини за даними комп’ютерної томографії у найважчих випадках досягав 80 відсотків.
5. Лабораторні показники крові демонструють суттєве підвищення рівня С-реактивного білка, що відображає наявність потужної системної запальної реакції.


In [23]:
print('ВАШЕ ПИТАННЯ:'); print(В['питання']); print()
ВІДПОВІДЬ = (
    "\nУ наборі даних яскравим прикладом стовпця з несумісними кодуваннями є «МНВ2» (а також «Креат1» та «МНВ1»), "
    "\nде основні дані записані як плаваючі числа (наприклад, 1.08, 1.18), проте через помилки ручного введення "
    "\nтрапляються грубі викиди-одруківки (наприклад, значення 239 замість 2.39) та сторонні типи даних "
    "\n(артефактні дати на кшталт 2015-01-01). Наївне обчислення середнього за допомогою стандартного переведення "
    "\nв числові типи через errors='coerce' призведе до втрати таких рядків або ж до катастрофічного спотворення "
    "\nрезультату через збереження масштабної помилки (викиду), яка зміщує середнє значення в кілька разів."
)

assert len(ВІДПОВІДЬ.strip()) >= 200, 'Відповідь має бути розгорнутою (від 200 символів)'
print(ВІДПОВІДЬ)

ВАШЕ ПИТАННЯ:
Знайдіть у файлі стовпець, де одна й та сама величина записана у двох несумісних кодуваннях. Скільки спостережень втратить наївне обчислення середнього?


У наборі даних яскравим прикладом стовпця з несумісними кодуваннями є «МНВ2» (а також «Креат1» та «МНВ1»), 
де основні дані записані як плаваючі числа (наприклад, 1.08, 1.18), проте через помилки ручного введення 
трапляються грубі викиди-одруківки (наприклад, значення 239 замість 2.39) та сторонні типи даних 
(артефактні дати на кшталт 2015-01-01). Наївне обчислення середнього за допомогою стандартного переведення 
в числові типи через errors='coerce' призведе до втрати таких рядків або ж до катастрофічного спотворення 
результату через збереження масштабної помилки (викиду), яка зміщує середнє значення в кілька разів.


---
# Етап 8. Декларація про використання інструментів ШІ

Користуватися мовними моделями дозволено. Це нормальна інженерна практика.

**Обов'язковою є прозорість:** ви вказуєте, де саме і як скористалися інструментом —
так само, як у науковій статті вказують використане програмне забезпечення.
Задекларована допомога порушенням не є.

Порушенням є інше: здати роботу, яку ви не можете пояснити. На захисті вас
попросять показати конкретний рядок свого коду, пояснити, чому він саме такий,
і внести в нього невелику зміну на місці.

Для кожного етапу оберіть одне з формулювань:

* `'ні'` — інструментами ШІ не користувався
* `'синтаксис'` — питав про назву функції або синтаксис, код писав сам
* `'код, перевірено'` — код згенеровано, я його прочитав, перевірив і розумію
* `'текст, перероблено'` — згенеровано чернетку тексту, яку я переписав по суті
* `'текст, як є'` — згенерований текст залишено майже без змін

In [25]:
ДЕКЛАРАЦІЯ = {
    'етап 1 — паспорт змінних':        'ні',
    'етап 2 — допустимі статистики':   'синтаксис',
    'етап 3 — безглузді числа':        'ні',
    'етап 4 — міра зв’язку':           'синтаксис',
    'етап 5 — підготовка до моделі':   'синтаксис',
    'етап 6 — виправлення аналізу':    'ні',
    'етап 7 — висновки й відповідь':   'ні',
}

ЩО_САМЕ_Я_ПЕРЕВІРИВ = 'Чому не можна використовувати кореляцію Пірсона до інших даних окрім Відношень x Відношень, а також запитувала щодо певних медичних показників - про що вони говорять і чи можемо ми з них зробити якийсь висновок щодо стану пацієнта.'
ПІДТВЕРДЖУЮ = True     # я можу пояснити кожен рядок свого коду
                        # і внести в нього зміни на вимогу під час захисту

In [26]:
ФОРМУЛЮВАННЯ = {'ні', 'синтаксис', 'код, перевірено', 'текст, перероблено', 'текст, як є'}

перевірки = {
    'заповнено титульні поля':        bool(ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР),
    'етап 1: паспорт із 8 стовпців':  all(ПАСПОРТ[c][0] and ПАСПОРТ[c][1] for c in МОЇ),
    'етап 3: три безглузді числа':    len(БЕЗГЛУЗДІ) >= 3,
    'етап 4: міру обрано й пояснено': bool(ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40),
    'етап 5: підготовку описано':     all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ),
    'етап 6: знайдено 10 помилок':    len(ВИПРАВЛЕННЯ) == 10,
    'етап 7: три висновки':           len(ВИСНОВКИ) >= 3,
    'етап 7: відповідь на питання':   len(ВІДПОВІДЬ.strip()) >= 200,
    'декларацію заповнено':           all(v in ФОРМУЛЮВАННЯ for v in ДЕКЛАРАЦІЯ.values()),
    'опис перевірки ≥ 120 символів':  len(ЩО_САМЕ_Я_ПЕРЕВІРИВ.strip()) >= 120,
    'підтвердження авторства':        ПІДТВЕРДЖУЮ is True,
}

print('ЧЕК-ЛИСТ ГОТОВНОСТІ')
print('=' * 52)
for назва, ок in перевірки.items():
    print(('  ГОТОВО     ' if ок else '  НЕ ГОТОВО ') + назва)
print('=' * 52)

assert all(перевірки.values()), 'Робота не готова до здачі — див. чек-лист вище'
print()
print(f'Робота варіанта {В["варіант"]} готова до захисту.')
print(f'{ПРІЗВИЩЕ_ІМЯ}, {ГРУПА}')

ЧЕК-ЛИСТ ГОТОВНОСТІ
  ГОТОВО     заповнено титульні поля
  ГОТОВО     етап 1: паспорт із 8 стовпців
  ГОТОВО     етап 3: три безглузді числа
  ГОТОВО     етап 4: міру обрано й пояснено
  ГОТОВО     етап 5: підготовку описано
  ГОТОВО     етап 6: знайдено 10 помилок
  ГОТОВО     етап 7: три висновки
  ГОТОВО     етап 7: відповідь на питання
  ГОТОВО     декларацію заповнено
  ГОТОВО     опис перевірки ≥ 120 символів
  ГОТОВО     підтвердження авторства

Робота варіанта 18 готова до захисту.
Щербан Ярина, ШІ-32


---
### Перед здачею

Виконайте **Kernel → Restart & Run All**. Ноутбук має пройти згори донизу
без жодної помилки — це перше, що перевіряється на захисті.